## Cell 1: Setup
**What this demonstrates**: Environment initialisation for Multi-Hop RAG — Haiku for fast bridge entity extraction at each hop, Sonnet for careful synthesis across the full reasoning chain. Separate timing is tracked per hop so latency accumulation is visible.
**Expected output**: Setup confirmation with model names, MAX_HOPS limit, and K_RETRIEVE per hop.

In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb anthropic openai python-dotenv

import os
import re
import json
import time
from dataclasses import dataclass, field

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

EXTRACT_MODEL = 'claude-haiku-4-5-20251001'  # Fast bridge entity extraction at each hop
SYNTH_MODEL   = 'claude-sonnet-4-6'           # Careful synthesis across full chain
EMBED_MODEL   = 'text-embedding-3-small'
MAX_HOPS      = 3    # Hard ceiling — never exceeded regardless of extraction result
K_RETRIEVE    = 3    # Documents retrieved per hop

client        = Anthropic()
lc_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Extract model : {EXTRACT_MODEL}  (bridge entity extraction per hop)')
print(f'  Synth model   : {SYNTH_MODEL}  (final answer from full chain)')
print(f'  MAX_HOPS      : {MAX_HOPS}  (hard ceiling — loop cannot exceed this)')
print(f'  K_RETRIEVE    : {K_RETRIEVE}  (docs per hop)')
print()
print('Key property: each hop\'s query is derived from the previous hop\'s result.')
print('The retrieval path is not known at query time — it unfolds dynamically.')

## Cell 2: Data — Synthetic Corporate Structure + Sanctions Corpus
**What this demonstrates**: A five-document corpus that forms an ownership chain: TechCorp Ltd → Nexus Holdings International → Viktor Sorin → OFAC SDN designation → BVI sanctions obligations. No single document contains the full picture — answering 'Is TechCorp connected to any sanctioned entities?' requires traversing all three intermediate hops. Each document is indexed as a single unit so retrieval boundaries are clean.
**Expected output**: Document count, source names, and a preview of the ownership chain embedded in the corpus.

In [ ]:
# Synthetic corpus — each document is a deliberate step in the ownership chain.
# The chain is: TechCorp → Nexus Holdings → Viktor Sorin → OFAC-designated.
# BVI and screening policy docs provide the regulatory answer at the chain's end.

CORPUS = {
    'techcorp_registration': """
TechCorp Ltd (Company No. TC-7821) is a technology services company incorporated
in the British Virgin Islands. The company provides software solutions to financial
institutions across Europe and Asia. TechCorp Ltd is a wholly-owned subsidiary of
Nexus Holdings International, which acquired the company in March 2021. Registered
office: Road Town, Tortola, BVI. Director: James Patterson. The company had revenues
of USD 14.2 million in the financial year ended 31 December 2023.
""".strip(),

    'nexus_holdings_structure': """
Nexus Holdings International (Registration No. NH-4492) is a holding company
incorporated in the Cayman Islands. The company holds interests in 12 subsidiary
entities across the technology and financial services sectors. Nexus Holdings
International is 80% owned by Viktor Sorin, a Ukrainian national born 15 March 1968.
The remaining 20% is held by Meridian Trust, a discretionary trust registered in
Jersey. Viktor Sorin serves as the non-executive chairman and ultimate beneficial
owner (UBO) of the Nexus Holdings group. Group assets under management: USD 340 million.
""".strip(),

    'ofac_sdn_excerpt': """
SORIN, Viktor (DOB: 15 March 1968; Nationality: Ukrainian; Passport: EM123456).
Added to OFAC SDN List: 14 January 2023. Basis: Executive Order 13661 — Blocking
Property of Additional Persons Contributing to the Situation in Ukraine. Related
entities: Nexus Holdings International, Cayman Islands; Meridian Trust, Jersey;
Sorin Capital Ltd, Cyprus. All transactions with Viktor Sorin, his immediate family
members, and entities he directly or indirectly owns or controls are prohibited under
US sanctions law. Non-US persons engaging in transactions with designated parties
may face secondary sanctions.
""".strip(),

    'bvi_sanctions_obligations': """
Under the Sanctions (British Virgin Islands) Act 2020 and the relevant UK Overseas
Territories Orders, entities incorporated in the British Virgin Islands are subject
to UK sanctions obligations. Any BVI-incorporated entity that is owned or controlled
directly or indirectly by a designated person must itself be treated as designated.
Ownership is defined as holding more than 50% of shares or voting rights. Financial
institutions dealing with BVI entities must conduct enhanced due diligence to identify
the ultimate beneficial owner and screen them against applicable sanctions lists.
""".strip(),

    'kyc_screening_policy': """
Financial institutions are required to screen all counterparties and their ultimate
beneficial owners (UBOs) against applicable sanctions lists including OFAC SDN,
UN Consolidated List, EU Consolidated List, and HM Treasury Financial Sanctions
List. Where a counterparty is owned or controlled by a sanctioned individual or
entity — even indirectly through a chain of ownership — the counterparty itself
must be treated as prohibited regardless of whether it appears on a sanctions list.
Failure to identify indirect sanctions exposure constitutes a compliance breach.
""".strip(),
}

# Build Chroma index — each document is kept whole (no chunking)
# so hop retrieval lands on a complete, coherent document
lc_docs = [
    Document(page_content=text, metadata={'source': name})
    for name, text in CORPUS.items()
]

print('Building corpus index...', end=' ', flush=True)
vectorstore = Chroma.from_documents(
    lc_docs, lc_embeddings, collection_name='multi_hop_rag_kyc'
)
print('done')
print(f'{len(lc_docs)} documents indexed')
print()
print('Ownership chain embedded in corpus:')
print('  techcorp_registration      → owned by Nexus Holdings International')
print('  nexus_holdings_structure   → 80% owned by Viktor Sorin (UBO)')
print('  ofac_sdn_excerpt           → Viktor Sorin: OFAC SDN-listed')
print('  bvi_sanctions_obligations  → BVI entity controlled by designated person = treated as designated')
print('  kyc_screening_policy       → indirect ownership through chain = prohibited counterparty')
print()
print('No single document contains the full picture.')
print('Answering the sanctions query requires traversing all three hops.')

## Cell 3: Core — Bridge Entity Extraction, Hop Loop, Answer Synthesis
**What this demonstrates**: Three functions implement Multi-Hop RAG. `extract_bridge_entity` uses Haiku to identify the next entity to retrieve (or signal terminal if the chain is complete). `multi_hop_rag` runs the hop loop — retrieve, extract, set new query, repeat — collecting a `list[HopResult]` path. `synthesise_answer` passes the full chain to Sonnet for a final answer with per-hop citations.
**Expected output**: Pipeline ready confirmation and a description of each hop component.

In [ ]:
# ── Shared data types ─────────────────────────────────────────────────────────

@dataclass
class ExtractionResult:
    """Output of extract_bridge_entity — the bridge to the next hop, or terminal signal."""
    bridge_entity: str    # Entity name to use as the next retrieval query
    entity_type  : str    # 'company', 'person', 'regulation', 'list', etc.
    reasoning    : str    # Why this entity bridges to the next piece of the answer
    terminal     : bool   # True if the accumulated context answers the original query
    answer_hint  : str    # Preliminary conclusion if terminal=True


@dataclass
class HopResult:
    """Complete record of one hop — query, retrieved docs, extraction, latency."""
    hop_number    : int
    query         : str
    docs_retrieved: list            # list[Document]
    extraction    : ExtractionResult
    latency_ms    : float


# ── Bridge entity extractor ───────────────────────────────────────────────────

def extract_bridge_entity(
    original_query : str,
    current_query  : str,
    docs           : list,
    hop_number     : int,
) -> ExtractionResult:
    """Identify the next entity to retrieve, or signal that the chain is complete.

    Uses EXTRACT_MODEL (Haiku) for speed. The model receives:
    - The original question (the destination we are trying to reach)
    - The current hop query (where we are now)
    - The retrieved documents (what this hop found)

    It returns a JSON object with the bridge entity and a terminal flag.

    Args:
        original_query: The user's initial question — unchanged across hops.
        current_query:  The query used for this hop (entity from prior hop).
        docs:           Documents retrieved in this hop.
        hop_number:     Current hop index (for context in the prompt).

    Returns:
        ExtractionResult with bridge entity or terminal=True.
    """
    context = '\n\n'.join(
        f'[{d.metadata.get("source", "doc")}]\n{d.page_content}' for d in docs
    )
    prompt = (
        f'Original question: {original_query}\n'
        f'Current hop query: {current_query}\n'
        f'Hop number: {hop_number}\n\n'
        f'Retrieved documents:\n{context}\n\n'
        'Task: Determine whether the retrieved documents contain enough information '
        'to answer the original question, OR identify the next entity to retrieve.\n\n'
        'Respond with a JSON object — no other text:\n'
        '{\n'
        '  "bridge_entity": "<name of next entity to look up, or empty string if terminal>",\n'
        '  "entity_type": "<company|person|regulation|list|other>",\n'
        '  "reasoning": "<one sentence: why this entity bridges to the next piece of the answer>",\n'
        '  "terminal": <true if original question can now be answered, false otherwise>,\n'
        '  "answer_hint": "<brief conclusion if terminal=true, else empty string>"\n'
        '}'
    )
    response = client.messages.create(
        model=EXTRACT_MODEL,
        max_tokens=300,
        messages=[{'role': 'user', 'content': prompt}],
    )
    raw = response.content[0].text.strip()
    # Strip markdown code fences if present
    raw = re.sub(r'```(?:json)?\s*', '', raw).strip('`').strip()
    match = re.search(r'\{[\s\S]*\}', raw)
    if match:
        try:
            parsed = json.loads(match.group())
            return ExtractionResult(
                bridge_entity = str(parsed.get('bridge_entity', '')).strip(),
                entity_type   = str(parsed.get('entity_type', 'unknown')),
                reasoning     = str(parsed.get('reasoning', '')),
                terminal      = bool(parsed.get('terminal', False)),
                answer_hint   = str(parsed.get('answer_hint', '')),
            )
        except (json.JSONDecodeError, ValueError):
            pass
    # Fallback: treat as not terminal, no bridge found — synthesiser will handle with what we have
    return ExtractionResult(
        bridge_entity='', entity_type='unknown',
        reasoning='Extraction parse failed — proceeding with accumulated context.',
        terminal=True, answer_hint='',
    )


# ── Answer synthesiser ────────────────────────────────────────────────────────

def synthesise_answer(original_query: str, path: list) -> str:
    """Generate a final answer from the full multi-hop reasoning chain.

    Each hop's documents and bridge entity reasoning are included in the
    context so the synthesiser can trace which hop contributed which fact.

    Args:
        original_query: The user's initial question.
        path:           list[HopResult] accumulated across all hops.

    Returns:
        Final answer string with hop-level citations.
    """
    chain_sections = []
    for hop in path:
        hop_docs = '\n'.join(
            f'  [{d.metadata.get("source", "doc")}]: {d.page_content[:350]}'
            for d in hop.docs_retrieved
        )
        chain_sections.append(
            f'=== Hop {hop.hop_number}: query="{hop.query}" ===\n'
            f'{hop_docs}\n'
            f'Bridge entity found: {hop.extraction.bridge_entity or "(none — terminal)"}\n'
            f'Reasoning: {hop.extraction.reasoning}'
        )
    full_chain = '\n\n'.join(chain_sections)
    response = client.messages.create(
        model=SYNTH_MODEL,
        max_tokens=500,
        system=(
            'You are a compliance analyst. Answer the question using only the provided '
            'multi-hop reasoning chain. Cite which hop (Hop 1, Hop 2, etc.) and which '
            'source document provided each key fact. Be precise about sanctions status.'
        ),
        messages=[{
            'role': 'user',
            'content': f'Question: {original_query}\n\nReasoning chain:\n{full_chain}',
        }],
    )
    return response.content[0].text.strip()


# ── Multi-hop loop ────────────────────────────────────────────────────────────

def multi_hop_rag(
    query      : str,
    vectorstore: Chroma,
    max_hops   : int = MAX_HOPS,
    k          : int = K_RETRIEVE,
) -> dict:
    """Multi-hop retrieval loop: retrieve → extract bridge → retrieve → ... → synthesise.

    The loop terminates when:
    - The extractor signals terminal=True (answer is reachable from accumulated context)
    - The extractor returns no bridge entity (chain is exhausted)
    - max_hops is reached (hard ceiling — always enforced)

    Args:
        query:       Initial user question.
        vectorstore: Chroma index over the corpus.
        max_hops:    Hard hop limit.
        k:           Documents retrieved per hop.

    Returns:
        dict with keys: query, path, hops_taken, answer, total_latency_ms.
    """
    path: list[HopResult] = []
    current_query = query
    t_total = time.perf_counter()

    for hop_num in range(1, max_hops + 1):
        t_hop = time.perf_counter()

        # Retrieve documents for the current query
        docs = vectorstore.similarity_search(current_query, k=k)

        # Extract bridge entity (or detect terminal condition)
        extraction = extract_bridge_entity(query, current_query, docs, hop_num)

        hop = HopResult(
            hop_number     = hop_num,
            query          = current_query,
            docs_retrieved = docs,
            extraction     = extraction,
            latency_ms     = (time.perf_counter() - t_hop) * 1000,
        )
        path.append(hop)

        # Terminate if answer is reachable or no bridge entity was found
        if extraction.terminal or not extraction.bridge_entity:
            break

        # Set bridge entity as the query for the next hop
        current_query = extraction.bridge_entity

    # Synthesise final answer from the complete reasoning chain
    t_synth = time.perf_counter()
    answer  = synthesise_answer(query, path)
    synth_ms = (time.perf_counter() - t_synth) * 1000

    return {
        'query'            : query,
        'path'             : path,
        'hops_taken'       : len(path),
        'answer'           : answer,
        'synth_latency_ms' : synth_ms,
        'total_latency_ms' : (time.perf_counter() - t_total) * 1000,
    }


print('Pipeline ready: multi_hop_rag(query, vectorstore)')
print()
print('Components:')
print(f'  extract_bridge_entity  — {EXTRACT_MODEL} — identifies next entity or terminal')
print(f'  multi_hop_rag loop     — max {MAX_HOPS} hops, {K_RETRIEVE} docs/hop')
print(f'  synthesise_answer      — {SYNTH_MODEL} — final answer with hop citations')

## Cell 4: Run — Sanctions Connectivity Query
**What this demonstrates**: A query that cannot be answered from a single retrieval. The initial query retrieves TechCorp's registration document. The extractor identifies Nexus Holdings International as the bridge. Hop 2 retrieves Nexus Holdings and identifies Viktor Sorin as the UBO bridge. Hop 3 retrieves Viktor Sorin and finds the OFAC designation — terminal condition met. The synthesiser assembles the full chain into a compliance answer.
**Expected output**: Per-hop summary (query, bridge entity found) and the final synthesised answer with hop citations.

In [ ]:
QUERY = 'Is TechCorp Ltd connected to any sanctioned entities?'

print(f'Query: {QUERY}')
print(f'Max hops: {MAX_HOPS}  |  Docs per hop: {K_RETRIEVE}')
print()

result = multi_hop_rag(QUERY, vectorstore)

# Per-hop summary
print(f'Hops taken: {result["hops_taken"]}')
print('-' * 65)
for hop in result['path']:
    ext    = hop.extraction
    srcs   = [d.metadata.get('source', '?') for d in hop.docs_retrieved]
    status = 'TERMINAL' if ext.terminal else f'→ "{ext.bridge_entity}"'
    print(f'Hop {hop.hop_number}  query="{hop.query[:55]}"')
    print(f'       docs : {", ".join(srcs)}')
    print(f'       bridge: {status}')
    if ext.reasoning:
        print(f'       reason: {ext.reasoning[:100]}')
    print()

print('Final answer:')
print('=' * 65)
print(result['answer'])
print()
print(f'Total latency: {result["total_latency_ms"]:.0f} ms  '
      f'(synthesis: {result["synth_latency_ms"]:.0f} ms)')

## Cell 5: Inspect — Hop-by-Hop Reasoning Chain
**What this demonstrates**: The full reasoning chain — documents retrieved at each hop, bridge entities extracted, and how the query evolved from the original question to the final sanctioned entity. A chain visualisation shows the ownership path. Timing breakdown confirms that latency scales with hop count.
**Expected output**: Detailed per-hop breakdown, chain visualisation, latency table, and comparison of what standard RAG (one hop) would have missed.

In [ ]:
# ── Per-hop detail ─────────────────────────────────────────────────────────────
for hop in result['path']:
    ext = hop.extraction
    print(f'HOP {hop.hop_number}')
    print('=' * 65)
    print(f'Query          : {hop.query}')
    print(f'Docs retrieved : {len(hop.docs_retrieved)}')
    for doc in hop.docs_retrieved:
        src = doc.metadata.get('source', 'unknown')
        print(f'  [{src}]')
        print(f'  {doc.page_content[:200].replace(chr(10), " ")}...')
    print()
    print(f'Bridge entity  : {ext.bridge_entity or "(none)"}')
    print(f'Entity type    : {ext.entity_type}')
    print(f'Reasoning      : {ext.reasoning}')
    print(f'Terminal       : {ext.terminal}')
    if ext.answer_hint:
        print(f'Answer hint    : {ext.answer_hint[:150]}')
    print(f'Hop latency    : {hop.latency_ms:.0f} ms')
    print()

# ── Chain visualisation ────────────────────────────────────────────────────────
print('REASONING CHAIN')
print('=' * 65)
chain_parts = [f'\'{result["query"][:30]}\'']
for hop in result['path']:
    if hop.extraction.bridge_entity:
        chain_parts.append(hop.extraction.bridge_entity)
    elif hop.extraction.terminal:
        chain_parts.append('[ANSWER FOUND]')
print('  →  '.join(chain_parts))
print()

# ── Latency breakdown ─────────────────────────────────────────────────────────
print('LATENCY BREAKDOWN')
print('=' * 65)
print(f'  {"Stage":30s}  {"ms":>6}')
print(f'  {"-"*30}  {"-"*6}')
for hop in result['path']:
    print(f'  {f"Hop {hop.hop_number} (retrieve + extract)":30s}  {hop.latency_ms:6.0f}')
print(f'  {"Synthesis (Sonnet)": 30s}  {result["synth_latency_ms"]:6.0f}')
print(f'  {"Total":30s}  {result["total_latency_ms"]:6.0f}')
print()

# ── What standard RAG would have returned ─────────────────────────────────────
print('STANDARD RAG BASELINE (hop 1 only)')
print('=' * 65)
if result['path']:
    hop1_docs = result['path'][0].docs_retrieved
    hop1_sources = [d.metadata.get('source', '?') for d in hop1_docs]
    print(f'Docs retrieved: {", ".join(hop1_sources)}')
    print(f'TechCorp registration is retrieved — but no sanctions connection is visible.')
    print(f'The bridge to Nexus Holdings is in the text, but standard RAG stops here.')
    print(f'Multi-Hop RAG continues for {result["hops_taken"] - 1} more hop(s) to reach the OFAC designation.')

## Cell 6: Fintech — KYC / UBO Resolution Chain
**What this demonstrates**: The same multi-hop mechanism applied to a KYC onboarding decision. A bank must determine whether it can safely onboard TechCorp Ltd as a client. This requires traversing the full ownership chain to identify the UBO and screen them against sanctions lists — the compliance answer is only reachable at the end of the chain. A second query demonstrates the pattern's reuse: the same pipeline, same corpus, different question, same chain traversal.
**Expected output**: Two KYC queries, their hop paths, and a side-by-side comparison of chain length and final compliance determination.

In [ ]:
KYC_QUERIES = [
    'What compliance obligations apply to a bank onboarding TechCorp Ltd as a new client?',
    'Who is the ultimate beneficial owner of TechCorp Ltd and what is their sanctions status?',
]

print('FINTECH: KYC ONBOARDING — MULTI-HOP OWNERSHIP RESOLUTION')
print('Use case: compliance officer assessing a new counterparty')
print()

kyc_results = []
for q in KYC_QUERIES:
    print(f'Running: "{q[:70]}"...', flush=True)
    r = multi_hop_rag(q, vectorstore)
    kyc_results.append(r)
    chain = ' → '.join(
        [q.split()[4]] +                              # First entity from query
        [h.extraction.bridge_entity for h in r['path'] if h.extraction.bridge_entity]
    )
    print(f'  Hops: {r["hops_taken"]}  |  {r["total_latency_ms"]:.0f} ms  |  chain: {chain}')
    print()

# ── Full answers ───────────────────────────────────────────────────────────────
for i, (q, r) in enumerate(zip(KYC_QUERIES, kyc_results)):
    print(f'QUERY {i+1}: {q}')
    print('=' * 65)
    print(r['answer'])
    print()

# ── Comparison table ───────────────────────────────────────────────────────────
print('KYC RESOLUTION SUMMARY')
print('=' * 65)
print(f'{"Query":50s}  {"Hops":5}  {"ms":>6}')
print(f'{"-"*50}  {"-"*5}  {"-"*6}')
for q, r in zip(KYC_QUERIES, kyc_results):
    print(f'{q[:48]:50s}  {r["hops_taken"]:5d}  {r["total_latency_ms"]:6.0f}')

print()
print('Chain traversed in both queries:')
print('  TechCorp Ltd  →  Nexus Holdings International  →  Viktor Sorin  →  OFAC SDN')
print()
print('A standard RAG system retrieves the TechCorp registration document and stops.')
print('The sanctions connection is invisible without following the ownership chain.')
print('Multi-Hop RAG surfaces it by treating each document as a step in a path,')
print('not an endpoint.')